In [0]:
from pyspark.sql import functions as f
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/adithyadasaadhiii@gmail.com/consolidated_pipeline/01_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sports-bar-raw/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

print(base_path)
print(landing_path)
print(processed_path)

bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.sb_fact_{data_source}'

In [0]:
df = (spark.read.options(Header = True, inferschema = True).csv(f'{landing_path}/*csv').withColumn("read_timestamp",f.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")
)

print("Total count :",df.count())
df.display(5)

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed","true").mode("append").saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)
files
for files_info in files:
    dbutils.fs.mv(
        files_info.path,
        f'{processed_path}/{files_info.name}',
        True
    )

In [0]:
df_orders = spark.read.table(bronze_table)
df_orders.show(1)

**Transformation in silver.**

In [0]:
#1.Order qty cannot be null if null drop the row
df_orders = df_orders.filter(f.col("order_qty").isNotNull())

#2.clean customer_id, keep numeric if not keep it as 999999
df_orders = df_orders.withColumn(
    "customer_id",
    f.when(f.col("customer_id").rlike("^[0-9]+$"), f.col("customer_id"))
    .otherwise(f.lit("999999" ))
    .cast("String")
)

#3remove weekday name from the date text
df_orders = df_orders.withColumn(
    "order_placement_date",
    f.regexp_replace("order_placement_date",r"^[A-Za-z]+,\s*","")
)
#4 convert date to date type
df_orders = df_orders.withColumn(
    "order_placement_date",
    f.coalesce(
        f.try_to_date("order_placement_date", "yyyy-MM-dd"),  # ISO (most common)
        f.try_to_date("order_placement_date", "yyyy/MM/dd"),
        f.try_to_date("order_placement_date", "dd-MM-yyyy"),
        f.try_to_date("order_placement_date", "dd/MM/yyyy"),
        f.try_to_date("order_placement_date", "d/M/yyyy"),
        f.try_to_date("order_placement_date", "yyyy-MM-dd HH:mm:ss"),
        f.try_to_date("order_placement_date", "yyyy-MM-dd'T'HH:mm:ss"),
        f.try_to_date("order_placement_date", "MMM dd, yyyy"),
        f.try_to_date("order_placement_date", "MMMM dd, yyyy")
    )
)
#5 drop duplicates
df_orders = df_orders.dropDuplicates(["order_id","order_placement_date","customer_id","product_id","order_qty"])

#6. Convert product_id to string
df_orders = df_orders.withColumn("product_id",f.col("product_id").cast("String"))


In [0]:
df_products = spark.read.table(f'{catalog}.{silver_schema}.products')
df_products.limit(5).show()

In [0]:
df_joined = df_orders.join(df_products,on="product_id",how="inner").select(df_orders["*"],df_products["product_code"])

display(df_joined.limit(10))

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Gold#

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

df_gold.show(2)

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

#Merging with Parent Data#

In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

In [0]:
df_child.count()

In [0]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", f.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        f.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


In [0]:
final_gold_fact_df = spark.read.table(f"{catalog}.{gold_schema}.fact_orders")
final_gold_fact_df.limit(5).display()